# 0. Import

In [1]:
!pip install wandb -q

In [2]:
import json
import wandb
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
from pathlib import Path


In [3]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hariswarsamasi (hariswarsamasi-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 1. Making the Modality

In [4]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [5]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [6]:
def normalize_skeleton(x):
    """
    x: (T, 17, 3)

    Makes skeleton coordinates root-relative.
    Joint 0 is treated as the root.
    """

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]
    x = x - root

    return x.astype(np.float32)

def temporal_resample(x, target_frames=64):
    """
    Resample the complete sequence to exactly target_frames.
    """

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    if T == 1:
        return np.repeat(
            x,
            target_frames,
            axis=0
        ).astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):

        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output.astype(np.float32)


## 3. Dataset Class

In [7]:
import json
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset


class SkeletonDataset(Dataset):

    def __init__(self, df, sequence_length=64):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    # --------------------------------------------------
    # Load skeleton
    # --------------------------------------------------

    def load_skeleton(self, path):

        prediction_dir = Path(path) / "predictions"

        json_files = sorted(prediction_dir.glob("*.json"))

        frames = []

        for json_file in json_files:

            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            person = data[0]

            keypoints = np.asarray(
                person["keypoints"],
                dtype=np.float32
            )

            frames.append(keypoints)

        if len(frames) == 0:
            return None

        return np.stack(frames)


    # --------------------------------------------------
    # NEW REPRESENTATION
    #
    # 1. Pelvis centering
    # 2. Scale normalization
    # --------------------------------------------------

    def normalize_skeleton(self, x):

        # x = (T, 17, 3)

        x = x.copy()

        # COCO-17 assumption:
        #
        # 5  = left shoulder
        # 6  = right shoulder
        # 11 = left hip
        # 12 = right hip

        left_hip = x[:, 11, :]
        right_hip = x[:, 12, :]

        left_shoulder = x[:, 5, :]
        right_shoulder = x[:, 6, :]

        # ----------------------------------------------
        # Pelvis / body center
        # ----------------------------------------------

        pelvis = (
            left_hip + right_hip
        ) / 2.0

        x = x - pelvis[:, None, :]

        # ----------------------------------------------
        # Torso scale
        #
        # distance between pelvis and shoulder center
        # ----------------------------------------------

        shoulder_center = (
            left_shoulder + right_shoulder
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center - pelvis,
            axis=1,
            keepdims=True
        )

        # Prevent division by zero
        scale = np.maximum(scale, 1e-6)

        x = x / scale[:, None, :]

        return x


    # --------------------------------------------------
    # Temporal resampling
    # --------------------------------------------------

    def temporal_resample(self, x):

        T = x.shape[0]

        if T == self.sequence_length:
            return x.astype(np.float32)

        if T == 1:

            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            ).astype(np.float32)

        old_indices = np.linspace(
            0,
            T - 1,
            T
        )

        new_indices = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        output = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for joint in range(x.shape[1]):

            for coord in range(x.shape[2]):

                output[:, joint, coord] = np.interp(
                    new_indices,
                    old_indices,
                    x[:, joint, coord]
                )

        return output


    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------

    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(
            row["skeleton"]
        )

        # ----------------------------------------------
        # Missing skeleton
        # ----------------------------------------------

        if skeleton is None:

            skeleton = np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

        # ----------------------------------------------
        # NEW:
        # pelvis centered + scale normalized
        # ----------------------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )
        # 3 + 3 + 3 = 9 features per joint
        skeleton = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration
            ],
            axis=2
        )

        # ----------------------------------------------
        # Resample AFTER derivatives
        # ----------------------------------------------

        skeleton = self.temporal_resample(
            skeleton
        )

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [8]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

MODALITIES = [
    "Skeleton",
    "Depth_Color",
    "IR",
    "Thermal",
    "IMU",
    "Radar",
]

# ---------------------------------------------------------
# Build trial index from Skeleton
# ---------------------------------------------------------

records = []

skeleton_root = TRAIN_DATA / "Skeleton"

for action_dir in sorted(skeleton_root.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name
    label = int(action_name.split("_")[0])

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
            })


train_df = pd.DataFrame(records)

print("Training trials:", len(train_df))
print("Classes:", train_df["label"].nunique())
print("Users:", train_df["user"].nunique())

train_df.head()

Training trials: 2931
Classes: 40
Users: 18


,action,label,user,trial,path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [9]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [10]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2238
Val dataset: 693


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [12]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 9])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 9])
Val batch y: torch.Size([32])


In [13]:
dataset =  SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)


X, y = dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 9])
y: tensor(0)
dtype: torch.float32
Min: -1.2548960447311401
Max: 1.2606968879699707
Mean: -0.03624916821718216
Std: 0.23697839677333832


---

In [14]:
X, y = train_dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 9])
y: tensor(0)
dtype: torch.float32
Min: -1.2548960447311401
Max: 1.2606968879699707
Mean: -0.03624916821718216
Std: 0.23697839677333832


In [15]:
print("Root joint mean:", X[:, 0, :].abs().mean().item())

print( "Root joint max:", X[:, 0, :].abs().max().item())

Root joint mean: 0.13719254732131958
Root joint max: 0.6045839190483093


# A. Baseline Model

In [16]:
import torch
import torch.nn as nn


class BiLSTMAttention(nn.Module):

    def __init__(
        self,
        input_size=153,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # 256 -> attention score
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # --------------------------------
        # (B, 64, 17, 6)
        # --------------------------------

        batch_size = x.size(0)

        # --------------------------------
        # Flatten joints + coordinates
        # --------------------------------

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        # (B, 64, 102)

        # --------------------------------
        # BiLSTM
        # --------------------------------

        output, _ = self.lstm(x)

        # (B, 64, 256)

        # --------------------------------
        # Attention
        # --------------------------------

        scores = self.attention(output)

        # (B, 64, 1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # --------------------------------
        # Weighted temporal representation
        # --------------------------------

        context = torch.sum(
            output * weights,
            dim=1
        )

        # (B, 256)

        # --------------------------------
        # Classification
        # --------------------------------

        logits = self.classifier(context)

        return logits

In [17]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BiLSTMAttention(
    input_size=153,
    hidden_size=128,
    num_layers=2,
    num_classes=40,
    dropout=0.3
).to(device)

print(model)
print("Device:", device)

BiLSTMAttention(
  (lstm): LSTM(153, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=40, bias=True)
  )
)
Device: cpu


In [18]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Input : torch.Size([32, 64, 17, 9])
Output: torch.Size([32, 40])
Train batch X: torch.Size([32, 64, 17, 9])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 9])
Val batch y: torch.Size([32])


In [19]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [20]:
wandb.init(
    project="CIUX",
    name="BiLstm with Velocity & Attention & Acceleration + Skeleton mapping",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 50,
        "optimizer": "Adam",
    }
)

In [22]:
epochs = 20
best_val_accuracy = 0.0
for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = output.argmax(dim=1)

        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            output = model(X)

            loss = criterion(output, y)

            val_loss += loss.item() * X.size(0)

            predictions = output.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_bilstm.pt"
        )

        print(
            f"🔥 New best model: "
            f"{best_val_accuracy:.4f}"
        )


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({
        "epoch": epoch + 1,

        "train/loss": train_loss,
        "train/accuracy": train_accuracy,

        "val/loss": val_loss,
        "val/accuracy": val_accuracy,

        "learning_rate": optimizer.param_groups[0]["lr"]
    })

🔥 New best model: 0.1587
Epoch 01/20 | Train Loss: 3.3959 | Train Acc: 0.1166 | Val Loss: 3.2028 | Val Acc: 0.1587
🔥 New best model: 0.2179
Epoch 02/20 | Train Loss: 3.0445 | Train Acc: 0.1747 | Val Loss: 2.9129 | Val Acc: 0.2179
🔥 New best model: 0.2799
Epoch 03/20 | Train Loss: 2.6817 | Train Acc: 0.2324 | Val Loss: 2.6957 | Val Acc: 0.2799
Epoch 04/20 | Train Loss: 2.4233 | Train Acc: 0.2797 | Val Loss: 2.6565 | Val Acc: 0.2684
Epoch 05/20 | Train Loss: 2.2453 | Train Acc: 0.3239 | Val Loss: 2.5177 | Val Acc: 0.2771
🔥 New best model: 0.2872
Epoch 06/20 | Train Loss: 2.1079 | Train Acc: 0.3503 | Val Loss: 2.4227 | Val Acc: 0.2872
🔥 New best model: 0.3203
Epoch 07/20 | Train Loss: 1.9547 | Train Acc: 0.3919 | Val Loss: 2.4260 | Val Acc: 0.3203
🔥 New best model: 0.3232
Epoch 08/20 | Train Loss: 1.8730 | Train Acc: 0.4106 | Val Loss: 2.4962 | Val Acc: 0.3232
🔥 New best model: 0.3680
Epoch 09/20 | Train Loss: 1.7684 | Train Acc: 0.4419 | Val Loss: 2.3623 | Val Acc: 0.3680
Epoch 10/20 | T

In [ ]:
wandb.finish()

---

# Z. Inference Part

## A. Test Dataset

In [24]:
class SkeletonTestDataset(SkeletonDataset):

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(
            row["skeleton"]
        )

        # No skeleton
        if skeleton is None:
            skeleton = np.zeros(
                (
                    self.sequence_length,
                    17,
                    3
                ),
                dtype=np.float32
            )

        # --------------------------------
        # SAME preprocessing as training
        # --------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )# Temporal velocity
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )
        # 3 + 3 + 3 = 9 features per joint
        skeleton = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration
            ],
            axis=2
        )
        skeleton = self.temporal_resample(
            skeleton
        )

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        return X, row["trial_id"]

## B. Defining the Path

In [25]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test")

for p in BASE.rglob("SM_test_0001"):
    print("Found:", p)

Found: /kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test/SM_test_0001


In [26]:
TEST_ROOT = p.parent
print(TEST_ROOT)

/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test


## 3. Testing Data

In [27]:
TEST_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Testing/Testing/"
    "small_model_track_test-007/small_model_track_test"
)

test_records = []

for trial_dir in sorted(TEST_ROOT.iterdir()):

    if not trial_dir.is_dir():
        continue

    # Ignore .claude or any non-test directory
    if not trial_dir.name.startswith("SM_test_"):
        continue

    skeleton_path = trial_dir / "Skeleton"

    if not skeleton_path.is_dir():
        continue

    test_records.append({
        "trial_id": trial_dir.name,
        "path": str(trial_dir),
        "skeleton": str(skeleton_path)
    })

test_df = pd.DataFrame(test_records)

print(test_df.head())
print("Number of test trials:", len(test_df))

       trial_id                                               path  \
0  SM_test_0001  /kaggle/input/datasets/samasiayushman/small-mo...   
1  SM_test_0002  /kaggle/input/datasets/samasiayushman/small-mo...   
2  SM_test_0003  /kaggle/input/datasets/samasiayushman/small-mo...   
3  SM_test_0004  /kaggle/input/datasets/samasiayushman/small-mo...   
4  SM_test_0005  /kaggle/input/datasets/samasiayushman/small-mo...   

                                            skeleton  
0  /kaggle/input/datasets/samasiayushman/small-mo...  
1  /kaggle/input/datasets/samasiayushman/small-mo...  
2  /kaggle/input/datasets/samasiayushman/small-mo...  
3  /kaggle/input/datasets/samasiayushman/small-mo...  
4  /kaggle/input/datasets/samasiayushman/small-mo...  
Number of test trials: 405


## 4. Calling the Dataset & Loader 

In [28]:
test_dataset = SkeletonTestDataset(
    test_df,
    sequence_length=64
)

print("Test trials:", len(test_dataset))

Test trials: 405


In [29]:
X, trial_id = test_dataset[0]

print("Trial:", trial_id)
print("Shape:", X.shape)
print("dtype:", X.dtype)
print("Min:", X.min().item())
print("Max:", X.max().item())

Trial: SM_test_0001
Shape: torch.Size([64, 17, 9])
dtype: torch.float32
Min: -1.1557648181915283
Max: 0.7105006575584412


In [30]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

X, trial_ids = next(iter(test_loader))

print("Batch X:", X.shape)
print("Trial IDs:", trial_ids[:5])

Batch X: torch.Size([32, 64, 17, 9])
Trial IDs: ('SM_test_0001', 'SM_test_0002', 'SM_test_0003', 'SM_test_0004', 'SM_test_0005')


## 5. Evaluating

In [31]:
model.load_state_dict(torch.load("/kaggle/working/best_bilstm.pt"))
model.eval()

all_predictions = []
all_trial_ids = []

with torch.no_grad():

    for X, trial_ids in test_loader:

        X = X.to(device)

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_trial_ids.extend(trial_ids)

## 6. Verifying Submission

In [33]:
print("Number of predictions:", len(all_predictions))
print("Number of trial IDs:", len(all_trial_ids))

print("\nFirst predictions:")
for trial_id, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):
    print(trial_id, "->", pred)

Number of predictions: 405
Number of trial IDs: 405

First predictions:
SM_test_0001 -> 36
SM_test_0002 -> 10
SM_test_0003 -> 10
SM_test_0004 -> 29
SM_test_0005 -> 20
SM_test_0006 -> 10
SM_test_0007 -> 26
SM_test_0008 -> 30
SM_test_0009 -> 37
SM_test_0010 -> 10


In [34]:
from collections import Counter

prediction_counts = Counter(all_predictions)

print("Predicted classes:")
for label, count in sorted(prediction_counts.items()):
    print(f"{label:2d}: {count}")

Predicted classes:
 0: 9
 1: 5
 2: 6
 3: 4
 4: 6
 5: 9
 6: 6
 7: 29
 8: 5
 9: 18
10: 29
11: 13
12: 6
13: 4
14: 1
15: 1
16: 1
17: 11
18: 6
19: 4
20: 19
21: 27
22: 5
23: 1
24: 12
26: 18
27: 5
28: 3
29: 12
30: 7
31: 9
32: 9
33: 5
34: 17
35: 3
36: 54
37: 12
38: 6
39: 8


## 7. Submission

In [35]:
submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head())
print(submission.shape)

submission.to_csv("submission.csv",index=False)

                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          10
2  small_model_track_test/SM_test_0003/          10
3  small_model_track_test/SM_test_0004/          29
4  small_model_track_test/SM_test_0005/          20
(405, 2)
